With random restarts, the same as ../3_1_pymc.ipynb

In [1]:
import os
import numpy as np
import pymc as pm
import pytensor
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
# if os.environ.get("SIMPLEVI_PYTENSOR_CXX"):
#     pytensor.config.cxx = os.environ["SIMPLEVI_PYTENSOR_CXX"]
pytensor.config.cxx = '/usr/bin/clang++'

In [2]:
mu_prior, sigma_prior = 0, 10
sigma_like = 1.0
max_iters = 1_600_000

from modulars import gaussian_1d_posterior, gaussian_1d
from modulars import run_pymc_VI, run_single_seed_pymc_VI

DATA = True
if DATA:
    data = gaussian_1d(100, 3, sigma_like, seed=0)
    true_mu_post, true_sigma_post = gaussian_1d_posterior(data, mu_prior=mu_prior, sigma_prior=sigma_prior, sigma_like=sigma_like)

else:
    data = []
    true_mu_post, true_sigma_post = mu_prior, sigma_prior

def run_model(n_mc_samples = 1, optimizer = 'default', seed = 1):
    with pm.Model() as model:
        # Define priors and likelihood
        mu = pm.Normal("mu", mu_prior, sigma_prior)
        likelihood = pm.Normal('y', mu=mu, sigma=sigma_like, observed=np.array(data))
        
        # run my pymc wrapper that allows for random restarts and 
        # tracking values across MC samples and optimizers
        tracked_vals = run_pymc_VI(model, n_mc_samples, max_iters, optimizer, seed)
    return tracked_vals

In [ ]:
results = []
adam_results = []
for seed in tqdm(range(50)):
    all_results = run_single_seed_pymc_VI(seed, run_model)
    results.append(all_results[0])
    adam_results.append(all_results[1])

  0%|          | 0/2 [00:00<?, ?it/s]

Finished [100%]: Average Loss = 147.34
Interrupted at 856,039 [53%]: Average Loss = 149.15


In [ ]:
from modulars import apply_traj_transform, save_rr_tracking_csv

TRACKING_CSV = "processed_tracking/rr_pymc_tracking.csv"

single_means, single_stds, multi_means, multi_stds = \
apply_traj_transform(results)
adam_single_means, adam_single_stds, adam_multi_means, adam_multi_stds = \
apply_traj_transform(adam_results)
save_rr_tracking_csv(
    TRACKING_CSV,
    {
        "default": (single_means, single_stds, multi_means, multi_stds),
        "adam": (adam_single_means, adam_single_stds, adam_multi_means, adam_multi_stds),
    },
)


In [ ]:
from modulars import load_rr_tracking_csv
from modulars import plot_a_few_trajectories_1d, plot_mean_band_rrs_1d

TRACKING_CSV = "processed_tracking/rr_pymc_tracking.csv"

single_means, single_stds, multi_means, multi_stds, x = load_rr_tracking_csv(
    TRACKING_CSV, scenario="default"
)
N, T = single_means.shape

plot_a_few_trajectories_1d(
    [single_means, single_stds], [multi_means, multi_stds],
    true_mu_post, true_sigma_post, r'$\mu$')
plot_mean_band_rrs_1d(
    single_means, single_stds, true_mu_post, true_sigma_post,
    x, r'$\mu$', 1)
plot_mean_band_rrs_1d(
    multi_means, multi_stds, true_mu_post, true_sigma_post,
    x, r'$\mu$', 100)


FileNotFoundError: [Errno 2] No such file or directory: 'processed_tracking/rr_pymc_tracking.csv'

# Adam results

In [ ]:
from modulars import load_rr_tracking_csv

TRACKING_CSV = "processed_tracking/rr_pymc_tracking.csv"

adam_single_means, adam_single_stds, adam_multi_means, adam_multi_stds, x = load_rr_tracking_csv(
    TRACKING_CSV, scenario="adam"
)
N, T = adam_single_means.shape

plot_a_few_trajectories_1d(
    [adam_single_means, adam_single_stds], [adam_multi_means, adam_multi_stds],
    true_mu_post, true_sigma_post, r'$\mu$', label_prefix = "(ADAM) ")
plot_mean_band_rrs_1d(
    adam_single_means, adam_single_stds, true_mu_post, true_sigma_post,
    x, r'$\mu$', 1, label_prefix = "(ADAM) ")
plot_mean_band_rrs_1d(
    adam_multi_means, adam_multi_stds, true_mu_post, true_sigma_post,
    x, r'$\mu$', 100, label_prefix = "(ADAM) ")
